In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Root Finding with Halley's Method

We already studied the Newton-Raphson method for finding roots: it was an *iterative* method that we applied over and over again to take a guess and improve it and improve it until we got very close to a zero.  We 'discovered' it by Taylor expanding
\begin{align}
    f(x+\Delta x) \approx f(x) + f'(x) \Delta x
\end{align}
and ignoring all terms higher-order in $\Delta x$.  Then, hoping to find a zero $f(x+\Delta x)=0$ and solving for $\Delta x$ one gets
\begin{align}
    \Delta x &= - \frac{f(x)}{f'(x)}
\end{align}
and then begins again with $x$ updated to $x + \Delta x$.

Here is a version of Newton-Raphson that doesn't just find the root but *also* tells you how many steps were required to find it.

In [ ]:
def newton_raphson(f, df, starting_guess, satisfying_smallness=1e-12):
    # starting_guess is our initial x_0
    # Let's check whether it's a zero (or close enough to satisfy our needs)
    
    guess = starting_guess
    height = f(guess)
    iterations = 0

    while satisfying_smallness < np.abs(height):
        slope = df(guess)
        change = - height / slope
        guess += change
        height = f(guess)
        iterations += 1
        
    return guess, iterations

Let's check how it performs when finding a root of $f(x) = x + e^x$.

In [ ]:
def f(x):
    return x + np.exp(x)

def df(x):
    return 1 + np.exp(x)

def ddf(x):
    return np.exp(x)

In [ ]:
x = np.linspace(-5,5,1000)
plt.plot(x, f(x))
plt.grid()

We'll study how many steps Newton-Raphson takes to find a root as a function of the starting point.

In [ ]:
starting_points = np.linspace(-10, 10, 1000)
root = np.zeros_like(starting_points)
nr_steps = np.zeros_like(starting_points)

for i, start in enumerate(starting_points):
    root[i], nr_steps[i] = newton_raphson(f, df, start)

fig, ax = plt.subplots()
ax.plot(starting_points, nr_steps, label='Newton-Raphson')
ax.set_title('Root-Finding Performance')
ax.set_xlabel('starting guess')
ax.set_ylabel('Steps required')
ax.legend()

But who told us to ignore all the higher-order terms in $\Delta x$?  That was our own choice, not a property of the function $f$ itself.

The Taylor expansion of $f$ might have an infinite number of terms---why stop at first order?

Unlike jokes, computational ideas can get better the more you tell them.

# Halley's Method

Edmund Halley had the same thought: maybe we can do better for just a little bit more work.

He said: don't just truncate the Taylor expansion at 1st order, go to 2nd order:
\begin{align}
    f(x + \Delta x) &= f(x) + f'(x) \Delta x + \frac{1}{2} f''(x) \Delta x^2 + \mathcal{O}(\Delta x^3).
\end{align}
This amounts to finding a *parabola* that goes through the point $(x, f(x))$ with exactly the right slope $f'(x)$ (as in Newton-Raphson) *and* just the right curvature $f''(x)$.

In [ ]:
x = np.linspace(-3,3)
plt.plot(x, f(x), label='f(x)')

x_0 = 0.
plt.plot(x, f(x_0) + df(x_0) * (x-x_0) + 1/2 * ddf(x_0) * (x-x_0)**2,
         linestyle='dashed',
         label='2nd order Taylor expansion',
        )

plt.legend()

Let's try to solve *that* approximation for $\Delta x$ while wishing for $f(x+\Delta x) = 0$ and neglecting the smaller mistakes ($\Delta x^3$ or smaller).

Up to the higher-order mistakes that we'll neglect, the equation is a quadratic equation in $\Delta x$!

So we can apply the quadratic formula to find
\begin{align}
    \Delta x &= \frac{-f'(x) \pm \sqrt{f'(x)^2 - 2 f''(x) f(x)}}{f''(x)}.
\end{align}

This formula is *correct* but it has some drawbacks.

1.  The quadratic formula has a $\pm$ in it.  Geometrically, if a parabola has real roots it will usually have 2 roots.  which choice should we make?
2.  We don't like to have square roots.  What if the contents of the square root are negative?  Geometrically: what if the tangent parabola has a turning point and doesn't actually go through 0?
3.  When we're near the right answer and $f(x)$ is very small then the square root in the numerator is very close to $f'(x)$ and so the numerator will have a big cancellation.  We've seen already that those kinds of big cancellations can cause a loss of precision, so we prefer to avoid that if possible.
4.  Even worse, this formula divides by $f''(x)$ by itself.  But when $f''(x)$ is very small the method ought to be almost equivalent to truncating the Taylor series to first order, but our formula would make $\Delta x$ very large without some good cancellations in the numerator.

The first quandry has a solution at least---we should try to pick the root nearest $\Delta x = 0$ since that will make our Taylor expansion more trustworthy.  The sign we should pick in $\Delta$ should be dictated by the sign of the slope.  If you look at the above figure it's clear that the 'near' zero is the one with the same-sign slope as we have at $x$.

This simplifies to
\begin{align}
    \Delta x &= \frac{-f'(x) + \text{sgn} f'(x) \times \sqrt{f'(x)^2 - 2 f''(x) f(x)}}{f''(x)}
\end{align}
where the `sgn` function is

In [ ]:
def sgn(x):
    if x < 0:
        return -1
    if x == 0:
        return 0
    if x > 0:
        return +1

The other issues can be addressed too, by remembering the philosophy: we already made a mistake of order $\Delta x^3$; if we make another error of that same size (or smaller) then we haven't made our method any worse.

The idea is as follows:
1.  Verify that \begin{align}
    \Delta x = \frac{-2 f(x)}{f'(x) + \text{sgn} f'(x) \times \sqrt{f'(x)^2 - 2 f''(x) f(x)}}
\end{align} is an exact ([though strange](https://en.wikipedia.org/wiki/Quadratic_formula#Square_root_in_the_denominator)!) rewriting of the roots we found above.
2.  Since we want a method that works when $f$ is small, Taylor expand the square root *with respect to f(x)*!
\begin{align}
\sqrt{f'(x)^2 - 2 f''(x) f(x)} \approx \sqrt{f'(x)^2} - \frac{f''(x) f(x)}{\sqrt{f'(x)^2}} + \cdots
\end{align} and therefore \begin{align}
    \Delta x \approx \frac{-2 f(x)}{f'(x) + \text{sgn} f'(x) \times \left(\sqrt{f'(x)^2} - \frac{f''(x) f(x)}{\sqrt{f'(x)^2}}\right)}.
\end{align}
3.  The square-roots give the absolute value $\sqrt{f'(x)^2} = | f'(x)|$.  But the square-root was multiplied by the sign of $f'(x)$ to begin with!  And the absolute value times the sign is just the thing itself,
$a = \text{sgn } a \times |a|$!  Therefore
\begin{align}
    \Delta x
    \approx \frac{-2 f(x)}{f'(x) + \left(f'(x) - \frac{f''(x) f(x)}{f'(x)}\right)}
    =
    -\frac{f(x) f'(x)}{f'(x)^2 - \frac{1}{2} f''(x) f(x)}.
\end{align}
This is a 'nicer' approximation because it has no square-root and doesn't have any subtraction that obviously requires delicate cancellations where we would be worried about precision problems.

Something interesting about this formula is that *if* $f''(x)=0$ it exactly reproduces Newton-Raphson.  In other words, for functions that don't bend much (so that $|f''(x)|$ is small in the region of the zero) the two methods should perform comparably.  This property wasn't straightforwardly obvious in our first expression where we just had $f''(x)$ alone in the denominator.

In [ ]:
def halley(f, df, ddf, starting_guess, satisfying_smallness=1e-12):
    # starting_guess is our initial x_0
    # satisfying_smallness is what we're willing to count as having found a zero.
    
    raise NotImplementedError("Please implement Halley's method!")
    return the_root, iterations

And let's compare your implementation to Newton-Raphson.

In [ ]:
root = np.zeros_like(starting_points)
h_steps = np.zeros_like(starting_points)

for i, start in enumerate(starting_points):
    root[i], h_steps[i] = halley(f, df, ddf, start)

fig, ax = plt.subplots()
ax.plot(starting_points, nr_steps, label='Newton-Raphson')
ax.plot(starting_points, h_steps, label='Halley')
ax.set_title('Root-Finding Performance')
ax.set_xlabel('starting guess')
ax.set_ylabel('Steps required')
ax.legend()

**Describe what you see!**

We can see that Halley's method differs from the Newton-Raphson method ...